# Propagation

> Where a transmission's channel coefficient h comes from: fading models and ray-traced tables.

In [ ]:
#| default_exp propagation

In [ ]:
#| hide
from nbdev.showdoc import *

A `Fading` gives the channel coefficient `h` of a transmission. `h` includes the path loss (`|h|²` is the power
gain) and the phase. Channels at both levels use it: `SystemLevelChannel` (see `system`) and `LinkLevelChannel`
(see `link`).

* `NoFading`: a constant `h`.
* `RayleighFading`: `h ~ CN(0, gain)`, drawn anew for every transmission.
* `RayTracedFading`: `h` looked up in a table ray-traced beforehand (e.g. with Sionna RT), at the nodes' positions.

In [ ]:
#| export
from __future__ import annotations

import math
import warnings
from abc import ABC, abstractmethod
from typing import Mapping, Sequence

import numpy as np

from comm_core.core import NodeId, Transmission
from comm_core.topology import Link

In [ ]:
#| export
class Fading(ABC):
    "The channel coefficient h of a transmission."

    @abstractmethod
    def coefficient(
        self,
        transmission: Transmission, # The transmission (its sender and receiver)
        link: Link, # Its link
    ) -> complex:
        "h of this transmission (math.inf: the nodes are at the same place, which means a perfect link)."

    def reset(self, seed: int | None = None) -> None:
        "Start over; `seed` reseeds the fading's random generator, if it has one."
        pass


class NoFading(Fading):
    "The same coefficient for every transmission."

    def __init__(
        self,
        h: complex = 1.0, # The coefficient
    ):
        self.h = h

    def coefficient(self, transmission, link):
        return self.h


class RayleighFading(Fading):
    "h ~ CN(0, gain), drawn anew for every transmission (constant during a packet)."

    def __init__(
        self,
        gain: float = 1.0, # Average power gain E|h|^2 (the path loss)
        seed: int | None = None, # Seed of the draws
    ):
        self.gain = gain
        self.rng = np.random.default_rng(seed)

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)

    def coefficient(self, transmission, link):
        return complex(self.rng.standard_normal(), self.rng.standard_normal()) * math.sqrt(self.gain / 2)

In [ ]:
show_doc(Fading.coefficient)

In [ ]:
t = Transmission(message=None, sender='a', receiver='b', start_time=0.0)
f = RayleighFading(gain=2.0, seed=0)
hs = np.array([f.coefficient(t, None) for _ in range(20_000)])
assert abs((np.abs(hs) ** 2).mean() - 2.0) < 0.05
f.reset(seed=0); assert f.coefficient(t, None) == hs[0]
assert NoFading(0.5j).coefficient(t, None) == 0.5j

### Ray-traced tables

The coefficients between every pair of a set of positions, e.g. the CFR from Sionna RT's `paths.cfr(...)`, saved
as numpy arrays:

* `H[rx, tx]`: complex, `(N, N)` or `(N, N, num_subcarriers)`.
* `positions[i]`: the position of index `i`, e.g. a grid cell `(x, y)`.

Nodes move, so the table is told where they are (`set_positions`, e.g. at every step of an environment). A
transmission gets the `H[rx, tx]` of its nodes' positions. Two nodes at the same position have a perfect link.
With subcarriers, a transmission gets one coefficient, `√(mean |H|²)`: the bits are not spread over the subcarriers.

The coefficients must keep the path loss, so compute them with `paths.cfr(..., normalize=False)`. With
`normalize=True` every link has `|H|² = 1` (only the phases differ) and every link would be equally good. The table
warns when it is given such an array.

In [ ]:
#| export
def _key(position) -> tuple:
    return tuple(np.asarray(position).reshape(-1).tolist())


class RayTracedFading(Fading):
    "Coefficients ray-traced beforehand (e.g. Sionna RT) between a set of positions, looked up where the nodes are."

    def __init__(
        self,
        H, # Channel coefficients `H[rx, tx]` between the positions: (N, N) or (N, N, num_subcarriers), complex
        positions: Sequence, # The N positions (e.g. grid cells (x, y)), in the order of H's indices
        node_positions: Mapping | None = None, # Where the nodes are to begin with (see `set_positions`)
    ):
        H = np.asarray(H)
        if H.ndim == 3:
            gain = (np.abs(H) ** 2).mean(axis=-1)
        elif H.ndim == 2:
            gain = np.abs(H) ** 2
        else:
            raise ValueError(f"H: (N, N) or (N, N, num_subcarriers), not {H.shape}")
        if gain.shape[0] != gain.shape[1] or gain.shape[0] != len(positions):
            raise ValueError(f"H is {H.shape[:2]} for {len(positions)} positions")
        self.gain = gain.astype(np.float64)
        self.H = (H if H.ndim == 2 else np.sqrt(self.gain)).astype(np.complex128)   # one coefficient per link
        self.positions = [_key(p) for p in positions]
        self.index = {p: i for i, p in enumerate(self.positions)}
        off = ~np.eye(len(self.positions), dtype=bool)
        if off.any() and np.allclose(self.gain[off], 1.0):
            warnings.warn("every link has |H|^2 = 1: the coefficients were normalized (paths.cfr(normalize=True)), "
                          "so the path loss is lost and every link is equally good; use normalize=False")
        self.node_positions: dict = {}
        self.set_positions(node_positions or {})

    @classmethod
    def from_files(
        cls,
        channel_path: str, # .npy of H[rx, tx]
        positions_path: str, # .npy of the positions
        **kwargs, # The other arguments of `RayTracedFading`
    ) -> RayTracedFading:
        "Load the arrays saved after ray tracing."
        return cls(np.load(channel_path), np.load(positions_path), **kwargs)

    def set_positions(
        self,
        positions: Mapping[NodeId, Sequence], # Position of (some of) the nodes, e.g. {'agent_0': (3, 5)}
    ) -> None:
        "Where the nodes are now."
        for node, p in positions.items():
            self.node_positions[node] = _key(p)

    def _index(self, node: NodeId) -> int:
        if node not in self.node_positions:
            raise KeyError(f"no position for node {node!r}: call set_positions first")
        p = self.node_positions[node]
        if p not in self.index:
            raise KeyError(f"node {node!r} is at {p}, where no channel was ray-traced ({len(self.index)} positions)")
        return self.index[p]

    def path_gain(self, sender: NodeId, receiver: NodeId) -> float:
        "Linear path gain |H|^2 from `sender` to `receiver` at their current positions (inf if co-located)."
        tx, rx = self._index(sender), self._index(receiver)
        return math.inf if tx == rx else float(self.gain[rx, tx])

    def coefficient(self, transmission, link):
        "H[rx, tx] at the nodes' current positions (math.inf if co-located)."
        tx, rx = self._index(transmission.sender), self._index(transmission.receiver)
        return math.inf if tx == rx else complex(self.H[rx, tx])

In [ ]:
show_doc(RayTracedFading.from_files)

In [ ]:
show_doc(RayTracedFading.set_positions)

In [ ]:
show_doc(RayTracedFading.path_gain)

In [ ]:
import tempfile, os

In [ ]:
positions = [(0, 0), (1, 0), (5, 0)]
H = np.array([[0, 1e-3, 2e-5], [1e-3j, 0, 3e-5], [2e-5, 3e-5, 0]])
table = RayTracedFading(H, positions, node_positions={'a': (0, 0), 'b': (1, 0)})
assert table.coefficient(t, None) == 1e-3j and abs(table.path_gain('a', 'b') - 1e-6) < 1e-18
table.set_positions({'b': (0, 0)})
assert table.coefficient(t, None) == math.inf and table.path_gain('a', 'b') == math.inf     # same cell
table.set_positions({'b': (9, 9)})
try:
    table.coefficient(t, None); raise AssertionError('no error')
except KeyError as e:
    assert 'no channel was ray-traced' in str(e)
H3 = np.repeat(H[..., None], 4, axis=-1)
assert np.allclose(RayTracedFading(H3, positions).gain, table.gain)
with tempfile.TemporaryDirectory() as tmp:
    np.save(os.path.join(tmp, 'H.npy'), H); np.save(os.path.join(tmp, 'pos.npy'), np.array(positions))
    assert RayTracedFading.from_files(os.path.join(tmp, 'H.npy'), os.path.join(tmp, 'pos.npy')).positions == positions
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    RayTracedFading(np.exp(1j * np.ones((3, 3))), positions)
    assert any('normalize' in str(x.message) for x in w)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()